# 02 Data Quality Audit

This notebook performs the audit-only quality sweep across all raw Olist tables.
No rows are cleaned or dropped in this step.

Phase 2: Data Quality Assessment & Exploratory Data Investigation

├── Dataset Inventory

├── Missing Value Analysis

├── Duplicate Analysis

├── Primary Key Validation

├── Composite Key Validation

├── Business Rule Investigation

├── Root Cause Analysis

├── Evidence-Based Investigation

└── Findings & Documentation

## import the packages 

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

## Step 2 — Load Every Dataset

In [3]:
customers = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_order_payments_dataset.csv")
products = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_products_dataset.csv")
reviews = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_order_reviews_dataset.csv")
sellers = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_sellers_dataset.csv")
geolocation = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_geolocation_dataset.csv")
translation = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/product_category_name_translation.csv")

## Step 3 — Store Them

In [4]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "products": products,
    "reviews": reviews,
    "sellers": sellers,
    "geolocation": geolocation,
    "translation": translation
}

## step 4 High Level Summary 

In [5]:
for name, df in datasets.items():
    print("=" * 60)
    print(name.upper())
    print("=" * 60)

    print(f"Rows    : {df.shape[0]}")
    print(f"Columns : {df.shape[1]}")
    print(f"Memory  : {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")

    print("\nData Types")
    print(df.dtypes)

    print("\n")

CUSTOMERS
Rows    : 99441
Columns : 5
Memory  : 26.59 MB

Data Types
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object


ORDERS
Rows    : 99441
Columns : 8
Memory  : 52.94 MB

Data Types
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


ORDER_ITEMS
Rows    : 112650
Columns : 7
Memory  : 35.99 MB

Data Types
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object


PAYMENTS
Rows    : 103886
Columns : 5
Memory  : 16.23 MB

Data

## step 5 Missing Value auditing

In [6]:
for name, df in datasets.items():

    missing = pd.DataFrame({
        "Missing Count": df.isnull().sum(),
        "Missing %": round(df.isnull().mean()*100,2)
    })

    missing = missing[missing["Missing Count"]>0]

    print("="*70)
    print(name.upper())

    if len(missing)==0:
        print("No Missing Values")
    else:
        print(missing.sort_values("Missing %",ascending=False))

CUSTOMERS
No Missing Values
ORDERS
                               Missing Count  Missing %
order_delivered_customer_date           2965       2.98
order_delivered_carrier_date            1783       1.79
order_approved_at                        160       0.16
ORDER_ITEMS
No Missing Values
PAYMENTS
No Missing Values
PRODUCTS
                            Missing Count  Missing %
product_category_name                 610       1.85
product_name_lenght                   610       1.85
product_description_lenght            610       1.85
product_photos_qty                    610       1.85
product_weight_g                        2       0.01
product_length_cm                       2       0.01
product_height_cm                       2       0.01
product_width_cm                        2       0.01
REVIEWS
                        Missing Count  Missing %
review_comment_title            87656      88.34
review_comment_message          58247      58.70
SELLERS
No Missing Values
GEOLOCATION
No Mi

## step 6 Duplicate checkups for primary key columns etc.

In [7]:
for name, df in datasets.items():

    duplicate_rows = df.duplicated().sum()

    print(f"{name:20} -> {duplicate_rows}")

customers            -> 0
orders               -> 0
order_items          -> 0
payments             -> 0
products             -> 0
reviews              -> 0
sellers              -> 0
geolocation          -> 261831
translation          -> 0


## STEP 7 PRIMARY KEY VALIDATION CHECKUP 

In [8]:
primary_keys = {
    "customers":"customer_id",
    "orders":"order_id",
    "products":"product_id",
    "reviews":"review_id",
    "sellers":"seller_id"
}

### run again the duplicate checkup 

In [9]:
for table,key in primary_keys.items():

    df = datasets[table]

    duplicates = df[key].duplicated().sum()

    print(f"{table:15} {key:20} duplicates = {duplicates}")

customers       customer_id          duplicates = 0
orders          order_id             duplicates = 0
products        product_id           duplicates = 0
reviews         review_id            duplicates = 814
sellers         seller_id            duplicates = 0


## STEP 8 CHECK COMPOSITE KEYS

In [10]:
order_items["order_id"].duplicated().sum()

np.int64(13984)

In [11]:
order_items.duplicated(
    subset=["order_id","order_item_id"]
).sum()

np.int64(0)

## STEP 9 CHECK COLUMN UNIQUENESS 

In [12]:
for name, df in datasets.items():

    print("="*60)

    print(name)

    summary = pd.DataFrame({

        "Unique Values":df.nunique(),

        "Total Rows":len(df)

    })

    summary["Unique %"] = round(
        summary["Unique Values"]/summary["Total Rows"]*100,
        2
    )

    print(summary)

customers
                          Unique Values  Total Rows  Unique %
customer_id                       99441       99441    100.00
customer_unique_id                96096       99441     96.64
customer_zip_code_prefix          14994       99441     15.08
customer_city                      4119       99441      4.14
customer_state                       27       99441      0.03
orders
                               Unique Values  Total Rows  Unique %
order_id                               99441       99441    100.00
customer_id                            99441       99441    100.00
order_status                               8       99441      0.01
order_purchase_timestamp               98875       99441     99.43
order_approved_at                      90733       99441     91.24
order_delivered_carrier_date           81018       99441     81.47
order_delivered_customer_date          95664       99441     96.20
order_estimated_delivery_date            459       99441      0.46
order_it

##  Step 10 — Value Counts (Categorical Inspection)

In [13]:
orders["order_status"].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [14]:
payments["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [15]:
customers["customer_state"].value_counts(dropna=False)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [16]:
products["product_category_name"].value_counts(dropna=False)

product_category_name
cama_mesa_banho                                   3029
esporte_lazer                                     2867
moveis_decoracao                                  2657
beleza_saude                                      2444
utilidades_domesticas                             2335
automotivo                                        1900
informatica_acessorios                            1639
brinquedos                                        1411
relogios_presentes                                1329
telefonia                                         1134
bebes                                              919
perfumaria                                         868
papelaria                                          849
fashion_bolsas_e_acessorios                        849
cool_stuff                                         789
ferramentas_jardim                                 753
pet_shop                                           719
NaN                                        

## STEP 11 NUMERIC SUMMARY 

In [17]:
for name, df in datasets.items():

    print("="*60)

    print(name)

    print(df.describe())

customers
       customer_zip_code_prefix
count              99441.000000
mean               35137.474583
std                29797.938996
min                 1003.000000
25%                11347.000000
50%                24416.000000
75%                58900.000000
max                99990.000000
orders
                                order_id                       customer_id  \
count                              99441                             99441   
unique                             99441                             99441   
top     e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
freq                                   1                                 1   

       order_status order_purchase_timestamp    order_approved_at  \
count         99441                    99441                99281   
unique            8                    98875                90733   
top       delivered      2018-03-31 15:08:21  2018-02-27 04:31:10   
freq          96478         

In [18]:
duplicate_reviews = reviews[
    reviews["review_id"].duplicated(keep=False)
]

duplicate_reviews.sort_values("review_id")

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados em dias diferentes, estão como em ""dificuldade na entrega"" segundo o rastreamento dos Correios. O que nunca me aconteceu antes.",2018-02-28 00:00:00,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07


In [19]:
duplicate_reviews.groupby("review_id").agg({
    "order_id": "count"
}).sort_values("order_id", ascending=False)

,order_id
review_id,
4219a80ab469e3fc9901437b73da3f75,3
7b606b0d57b078384f0b58eac1d41d78,3
0c76e7a547a531e7bf9f0b99cba071c1,3
69a1068c3128a14994e3e422e4539e04,3
308316408775d1600dad81bd3184556d,3
...,...
fde2e6abaf5bb64f7407a44741c24dec,2
fde5986d35c89aa1b6ce4149de82a0d3,2
fe5c833752953fed3209646f1f63b53c,2


In [22]:
review = reviews[
    reviews["review_id"]=="4219a80ab469e3fc9901437b73da3f75"
]

review

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
5318,4219a80ab469e3fc9901437b73da3f75,56ae029ed8bd31758f66831a64bf5629,5,NaN,Muito rápido a entrega!!!,2017-05-18 00:00:00,2017-05-19 19:12:36
55335,4219a80ab469e3fc9901437b73da3f75,84f5e6c0a0e3155e38c00f434ba90ce8,5,NaN,Muito rápido a entrega!!!,2017-05-18 00:00:00,2017-05-19 19:12:36
63704,4219a80ab469e3fc9901437b73da3f75,d9e44c3fd2ce16086619f299e92e12d8,5,NaN,Muito rápido a entrega!!!,2017-05-18 00:00:00,2017-05-19 19:12:36


### CHECK ARE THESE 3 ORDERS FROM SAME CUSTOMER?

In [ ]:
orders.loc[ orders["order_id"].isin([
        "56ae029ed8bd31758f66831a64bf5629",
        "84f5e6c0a0e3155e38c00f434ba90ce8",
        "d9e44c3fd2ce16086619f299e92e12d8"
    ]),
    ["order_id","customer_id","order_purchase_timestamp"]
]

,order_id,customer_id,order_purchase_timestamp
22179,56ae029ed8bd31758f66831a64bf5629,4819e20338590ee062fedff6ebde9167,2017-05-06 20:11:10
58749,d9e44c3fd2ce16086619f299e92e12d8,213573df8c484028f3492fa3548074da,2017-05-06 20:11:10
59312,84f5e6c0a0e3155e38c00f434ba90ce8,0cecd74a34f636f301b750123dd4dd3c,2017-05-06 20:11:11


### we are checking whether 

One person can have multiple customer IDs.
One customer can have multiple orders.
One review can be associated with multiple orders.
A duplicate key isn't automatically bad.

In [24]:
sample_customer_ids = [
    "4819e20338590ee062fedff6ebde9167",
    "213573df8c484028f3492fa3548074da",
    "0cecd74a34f636f301b750123dd4dd3c"
]

customers.loc[
    customers["customer_id"].isin(sample_customer_ids),
    ["customer_id", "customer_unique_id"]
]

,customer_id,customer_unique_id
34647,213573df8c484028f3492fa3548074da,06a52782a04f0086d16b9c22d0e29438
62472,0cecd74a34f636f301b750123dd4dd3c,06a52782a04f0086d16b9c22d0e29438
94798,4819e20338590ee062fedff6ebde9167,06a52782a04f0086d16b9c22d0e29438
